# 10.2 Cloud Deployment — Apply

## Objective

Build production-grade ONNX inference components suitable for cloud deployment with FastAPI.
We construct request handlers, introspect model metadata, implement health checks,
generate Kubernetes manifests, and measure throughput — all without starting an actual server.

**Prerequisites:** `pip install onnx onnxruntime numpy`

## Table of Contents
1. [Setup](#setup)
2. [Exercise 1 — Production ORT Session](#ex1)
3. [Exercise 2 — Session Metadata Introspection](#ex2)
4. [Exercise 3 — FastAPI-Compatible Request Handler](#ex3)
5. [Exercise 4 — Tensor Preprocessing & Postprocessing](#ex4)
6. [Exercise 5 — Kubernetes Deployment Manifest Generator](#ex5)
7. [Exercise 6 — Health Check & Readiness Probes](#ex6)
8. [Exercise 7 — Load Testing Simulation](#ex7)
9. [Challenge — Complete Inference API Skeleton](#challenge)
10. [Summary](#summary)

In [ ]:
!pip install onnx onnxruntime numpy -q

<a id='setup'></a>
## Setup

We create a small ONNX model (linear regression) to use as our deployment target throughout
this notebook. In production you would load a real model; here we keep it minimal so every
cell runs instantly.

In [ ]:
import os
import time
import json
import numpy as np
from typing import Any, Dict, List, Optional, Tuple

import onnx
from onnx import helper, TensorProto, numpy_helper
import onnxruntime as ort

# Build a small model: Y = X @ W + B (linear layer with 4 inputs, 3 outputs)
np.random.seed(42)
W_val = np.random.randn(4, 3).astype(np.float32)
B_val = np.random.randn(3).astype(np.float32)

X = helper.make_tensor_value_info("input", TensorProto.FLOAT, ["batch", 4])
Y = helper.make_tensor_value_info("output", TensorProto.FLOAT, ["batch", 3])

W_init = numpy_helper.from_array(W_val, name="W")
B_init = numpy_helper.from_array(B_val, name="B")

matmul = helper.make_node("MatMul", ["input", "W"], ["XW"])
add = helper.make_node("Add", ["XW", "B"], ["output"])

graph = helper.make_graph([matmul, add], "linear_model", [X], [Y], [W_init, B_init])
model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
model.ir_version = 8
onnx.checker.check_model(model)

MODEL_PATH = "/tmp/cloud_deploy_model.onnx"
onnx.save(model, MODEL_PATH)
print(f"Model saved to {MODEL_PATH} ({os.path.getsize(MODEL_PATH)} bytes)")

<a id='ex1'></a>
## Exercise 1 — Production ORT Session

A production session should:
- Enable all graph optimizations
- Set appropriate thread counts (avoid over-subscription)
- Select execution providers deliberately

The **intra-op parallelism** degree controls parallelism within a single operator:

$$\text{Latency} \approx \frac{\text{FLOPs}}{\text{threads} \times \text{FLOPS\_per\_thread}}$$

Setting `intra_op_num_threads = 0` lets ORT use all cores — good for throughput, risky
for latency SLAs under concurrent requests.

In [ ]:
def build_production_session(
    model_path: str,
    intra_op_threads: int = 2,
    inter_op_threads: int = 1,
    optimization_level: str = "all",
) -> ort.InferenceSession:
    """Build an ORT session with production settings."""
    so = ort.SessionOptions()

    opt_map = {
        "all": ort.GraphOptimizationLevel.ORT_ENABLE_ALL,
        "basic": ort.GraphOptimizationLevel.ORT_ENABLE_BASIC,
        "extended": ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED,
        "none": ort.GraphOptimizationLevel.ORT_DISABLE_ALL,
    }
    so.graph_optimization_level = opt_map.get(optimization_level, ort.GraphOptimizationLevel.ORT_ENABLE_ALL)
    so.intra_op_num_threads = intra_op_threads
    so.inter_op_num_threads = inter_op_threads
    so.enable_mem_pattern = True
    so.enable_cpu_mem_arena = True

    providers = []
    available = ort.get_available_providers()
    if "CUDAExecutionProvider" in available:
        providers.append("CUDAExecutionProvider")
    providers.append("CPUExecutionProvider")

    session = ort.InferenceSession(model_path, sess_options=so, providers=providers)
    return session


session = build_production_session(MODEL_PATH)
print("Active providers:", session.get_providers())

# Verify inference works
test_input = np.random.randn(1, 4).astype(np.float32)
result = session.run(None, {"input": test_input})[0]
expected = test_input @ W_val + B_val
assert np.allclose(result, expected, atol=1e-5), "Session output mismatch!"
print(f"Inference OK — output shape: {result.shape}")

<a id='ex2'></a>
## Exercise 2 — Session Metadata Introspection

Before serving a model, we must know its expected inputs and outputs.
This information drives:
- Input validation in the request handler
- API schema generation (OpenAPI / JSON Schema)
- Client SDK code generation

In [ ]:
def introspect_session(sess: ort.InferenceSession) -> Dict[str, Any]:
    """Extract comprehensive metadata from an ORT session."""
    def tensor_info(t) -> Dict[str, Any]:
        return {
            "name": t.name,
            "shape": list(t.shape),
            "type": t.type,
            "numpy_dtype": {
                "tensor(float)": "float32",
                "tensor(double)": "float64",
                "tensor(int64)": "int64",
                "tensor(int32)": "int32",
            }.get(t.type, "unknown"),
        }

    meta = sess.get_modelmeta()
    return {
        "inputs": [tensor_info(i) for i in sess.get_inputs()],
        "outputs": [tensor_info(o) for o in sess.get_outputs()],
        "providers": sess.get_providers(),
        "model_metadata": {
            "producer": meta.producer_name,
            "graph_name": meta.graph_name,
            "description": meta.description,
            "domain": meta.domain,
            "version": meta.version,
        },
    }


metadata = introspect_session(session)
print(json.dumps(metadata, indent=2))

# Assertions
assert len(metadata["inputs"]) == 1
assert metadata["inputs"][0]["name"] == "input"
assert metadata["inputs"][0]["numpy_dtype"] == "float32"
assert len(metadata["outputs"]) == 1
assert metadata["outputs"][0]["name"] == "output"
print("\nMetadata introspection validated.")

<a id='ex3'></a>
## Exercise 3 — FastAPI-Compatible Request Handler

We build a handler function matching the interface of a FastAPI route:
- Accepts a request dict (simulating Pydantic model)
- Validates and reshapes input tensors
- Returns JSON-serializable response

We test it via direct function calls (no HTTP server needed).

In [ ]:
class InferenceError(Exception):
    """Custom exception for inference failures with HTTP-like status codes."""
    def __init__(self, status_code: int, detail: str):
        self.status_code = status_code
        self.detail = detail
        super().__init__(detail)


def handle_inference_request(
    sess: ort.InferenceSession,
    request: Dict[str, Any],
) -> Dict[str, Any]:
    """
    FastAPI-compatible inference handler.
    
    Request format:
        {"data": [...], "shape": [batch, features], "dtype": "float32", "input_name": optional}
    """
    # Validate required fields
    if "data" not in request or "shape" not in request:
        raise InferenceError(400, "Missing 'data' or 'shape' in request")

    dtype_str = request.get("dtype", "float32")
    try:
        dtype = np.dtype(dtype_str)
    except TypeError as e:
        raise InferenceError(400, f"Invalid dtype: {e}")

    try:
        arr = np.asarray(request["data"], dtype=dtype).reshape(request["shape"])
    except (ValueError, TypeError) as e:
        raise InferenceError(400, f"Cannot reshape data: {e}")

    # Determine input name
    inputs = sess.get_inputs()
    input_name = request.get("input_name", inputs[0].name)

    # Run inference
    output_names = [o.name for o in sess.get_outputs()]
    try:
        outputs = sess.run(output_names, {input_name: arr})
    except Exception as e:
        raise InferenceError(500, f"Inference failed: {e}")

    # Build response
    results = []
    for name, out in zip(output_names, outputs):
        out_np = np.asarray(out)
        results.append({
            "name": name,
            "shape": list(out_np.shape),
            "dtype": str(out_np.dtype),
            "data": out_np.ravel().tolist(),
        })

    return {"outputs": results}


# Test the handler with valid request
valid_request = {
    "data": np.random.randn(2, 4).ravel().tolist(),
    "shape": [2, 4],
    "dtype": "float32",
}
response = handle_inference_request(session, valid_request)
assert response["outputs"][0]["shape"] == [2, 3]
print("Valid request response:", json.dumps(response["outputs"][0]["shape"]))

# Test with invalid request
try:
    handle_inference_request(session, {"data": [1, 2], "shape": [5, 5]})
    assert False, "Should have raised"
except InferenceError as e:
    assert e.status_code == 400
    print(f"Correctly caught bad request: {e.detail}")

<a id='ex4'></a>
## Exercise 4 — Tensor Preprocessing & Postprocessing

Real APIs require preprocessing (normalization, type casting) and postprocessing
(softmax, argmax, label mapping). We build reusable components.

**Softmax** converts logits $z$ to probabilities:

$$\text{softmax}(z_i) = \frac{e^{z_i - \max(z)}}{\sum_j e^{z_j - \max(z)}}$$

The $\max(z)$ subtraction prevents numerical overflow in `exp`.

In [ ]:
def preprocess_input(
    raw_data: List[float],
    shape: List[int],
    dtype: str = "float32",
    normalize: bool = False,
    mean: Optional[np.ndarray] = None,
    std: Optional[np.ndarray] = None,
) -> np.ndarray:
    """Convert raw request data to a validated numpy tensor."""
    arr = np.asarray(raw_data, dtype=np.dtype(dtype)).reshape(shape)

    if normalize and mean is not None and std is not None:
        arr = (arr - mean) / std

    return arr


def softmax(logits: np.ndarray, axis: int = -1) -> np.ndarray:
    """Numerically stable softmax."""
    shifted = logits - np.max(logits, axis=axis, keepdims=True)
    exp_vals = np.exp(shifted)
    return exp_vals / np.sum(exp_vals, axis=axis, keepdims=True)


def postprocess_classification(
    logits: np.ndarray,
    labels: Optional[List[str]] = None,
    top_k: int = 3,
) -> Dict[str, Any]:
    """Convert raw logits to top-k classification results."""
    probs = softmax(logits)
    top_indices = np.argsort(-probs, axis=-1)[..., :top_k]

    results = []
    for batch_idx in range(probs.shape[0]):
        batch_results = []
        for idx in top_indices[batch_idx]:
            entry = {"index": int(idx), "probability": float(probs[batch_idx, idx])}
            if labels:
                entry["label"] = labels[idx] if idx < len(labels) else f"class_{idx}"
            batch_results.append(entry)
        results.append(batch_results)

    return {"predictions": results}


# Test preprocessing
raw = np.random.randn(2, 4).ravel().tolist()
tensor = preprocess_input(raw, [2, 4])
assert tensor.shape == (2, 4)
assert tensor.dtype == np.float32

# Test postprocessing
logits = session.run(None, {"input": tensor})[0]
labels = ["cat", "dog", "bird"]
result = postprocess_classification(logits, labels=labels, top_k=2)
assert len(result["predictions"]) == 2
assert len(result["predictions"][0]) == 2
assert sum(p["probability"] for p in result["predictions"][0]) <= 1.0 + 1e-5
print("Postprocessing result:", json.dumps(result, indent=2))

<a id='ex5'></a>
## Exercise 5 — Kubernetes Deployment Manifest Generator

Production cloud deployments use Kubernetes. We generate a deployment + service
manifest programmatically. Key decisions:

- **Resource requests/limits**: prevent OOM kills and noisy neighbors
- **Replicas**: horizontal scaling for throughput
- **Liveness/readiness probes**: Kubernetes health management

In [ ]:
def generate_k8s_manifest(
    model_name: str,
    image: str,
    replicas: int = 2,
    cpu_request: str = "500m",
    cpu_limit: str = "2000m",
    memory_request: str = "512Mi",
    memory_limit: str = "2Gi",
    port: int = 8000,
    model_path: str = "/models/model.onnx",
    intra_op_threads: int = 2,
) -> str:
    """Generate a Kubernetes Deployment + Service YAML manifest."""
    manifest = f"""apiVersion: apps/v1
kind: Deployment
metadata:
  name: {model_name}-deployment
  labels:
    app: {model_name}
spec:
  replicas: {replicas}
  selector:
    matchLabels:
      app: {model_name}
  template:
    metadata:
      labels:
        app: {model_name}
    spec:
      containers:
      - name: {model_name}
        image: {image}
        ports:
        - containerPort: {port}
        env:
        - name: MODEL_PATH
          value: "{model_path}"
        - name: INTRA_OP_THREADS
          value: "{intra_op_threads}"
        - name: PORT
          value: "{port}"
        resources:
          requests:
            cpu: "{cpu_request}"
            memory: "{memory_request}"
          limits:
            cpu: "{cpu_limit}"
            memory: "{memory_limit}"
        livenessProbe:
          httpGet:
            path: /health
            port: {port}
          initialDelaySeconds: 10
          periodSeconds: 30
        readinessProbe:
          httpGet:
            path: /health
            port: {port}
          initialDelaySeconds: 5
          periodSeconds: 10
---
apiVersion: v1
kind: Service
metadata:
  name: {model_name}-service
spec:
  selector:
    app: {model_name}
  ports:
  - protocol: TCP
    port: 80
    targetPort: {port}
  type: ClusterIP
"""
    return manifest


manifest = generate_k8s_manifest(
    model_name="onnx-linear",
    image="myregistry/onnx-server:v1.0",
    replicas=3,
)
print(manifest)

# Validate manifest contains expected fields
assert "replicas: 3" in manifest
assert "onnx-linear-deployment" in manifest
assert "readinessProbe" in manifest
assert "livenessProbe" in manifest
assert "MODEL_PATH" in manifest
print("Manifest validation passed.")

<a id='ex6'></a>
## Exercise 6 — Health Check & Readiness Probes

Health endpoints serve two purposes in cloud deployments:

| Probe | Purpose | Failure Action |
|-------|---------|----------------|
| **Liveness** | Is the process alive? | Restart container |
| **Readiness** | Can it accept traffic? | Remove from load balancer |

Our readiness check verifies the model can actually run inference (warm-up inference).

In [ ]:
def liveness_check() -> Dict[str, Any]:
    """Basic liveness: process is responsive."""
    return {"status": "alive", "timestamp": time.time()}


def readiness_check(
    sess: Optional[ort.InferenceSession],
    model_path: str,
) -> Dict[str, Any]:
    """Deep readiness: model loaded AND can run inference."""
    if sess is None:
        return {"status": "not_ready", "reason": "session not loaded"}

    if not os.path.isfile(model_path):
        return {"status": "not_ready", "reason": "model file missing"}

    # Attempt a warm-up inference with minimal input
    try:
        inp = sess.get_inputs()[0]
        shape = [1 if isinstance(d, str) else d for d in inp.shape]
        dummy = np.zeros(shape, dtype=np.float32)
        sess.run(None, {inp.name: dummy})
    except Exception as e:
        return {"status": "not_ready", "reason": f"inference failed: {e}"}

    return {
        "status": "ready",
        "model_path": model_path,
        "providers": sess.get_providers(),
        "num_inputs": len(sess.get_inputs()),
        "num_outputs": len(sess.get_outputs()),
    }


# Test liveness
live = liveness_check()
assert live["status"] == "alive"
assert "timestamp" in live

# Test readiness — should pass with our session
ready = readiness_check(session, MODEL_PATH)
assert ready["status"] == "ready"
print("Readiness:", json.dumps(ready, indent=2))

# Test readiness failure — None session
not_ready = readiness_check(None, MODEL_PATH)
assert not_ready["status"] == "not_ready"
print(f"Not ready (no session): {not_ready['reason']}")

<a id='ex7'></a>
## Exercise 7 — Load Testing Simulation

Measure throughput and latency distribution to estimate capacity planning.

Key metrics:
- **P50/P95/P99 latency**: tail latency matters for SLAs
- **Throughput** (requests/sec): determines replica count
- **Capacity formula**:

$$\text{replicas} = \left\lceil \frac{\text{target\_RPS} \times P_{99}}{\text{concurrency\_per\_pod}} \right\rceil$$

In [ ]:
def run_load_test(
    sess: ort.InferenceSession,
    num_requests: int = 500,
    batch_size: int = 1,
) -> Dict[str, Any]:
    """Simulate load testing by measuring sequential inference latencies."""
    inp = sess.get_inputs()[0]
    shape = [batch_size if isinstance(d, str) else d for d in inp.shape]
    test_data = np.random.randn(*shape).astype(np.float32)

    latencies = []
    for _ in range(num_requests):
        start = time.perf_counter()
        sess.run(None, {inp.name: test_data})
        latencies.append(time.perf_counter() - start)

    latencies_ms = np.array(latencies) * 1000

    total_time = sum(latencies)
    throughput = num_requests / total_time

    return {
        "num_requests": num_requests,
        "batch_size": batch_size,
        "total_time_s": round(total_time, 3),
        "throughput_rps": round(throughput, 1),
        "latency_ms": {
            "mean": round(float(np.mean(latencies_ms)), 3),
            "p50": round(float(np.percentile(latencies_ms, 50)), 3),
            "p95": round(float(np.percentile(latencies_ms, 95)), 3),
            "p99": round(float(np.percentile(latencies_ms, 99)), 3),
            "max": round(float(np.max(latencies_ms)), 3),
        },
    }


results = run_load_test(session, num_requests=1000, batch_size=1)
print(json.dumps(results, indent=2))

# Verify reasonable performance
assert results["throughput_rps"] > 100, "Throughput too low for a tiny model"
assert results["latency_ms"]["p50"] < 50, "P50 latency unexpectedly high"

# Capacity calculation example
target_rps = 10000
p99 = results["latency_ms"]["p99"] / 1000  # convert to seconds
concurrency_per_pod = 4
import math
replicas_needed = math.ceil((target_rps * p99) / concurrency_per_pod)
print(f"\nFor {target_rps} RPS target with P99={p99*1000:.1f}ms:")
print(f"  Estimated replicas needed: {replicas_needed}")

<a id='challenge'></a>
## Challenge — Complete Inference API Server Skeleton

Combine all components into a cohesive API server class that could be
wrapped by FastAPI. Implement:

1. Model loading with error handling
2. Health/readiness endpoints
3. Inference with pre/post processing
4. Request logging and metrics collection

In [ ]:
class ONNXInferenceServer:
    """Production inference server skeleton (no HTTP, just handler logic)."""

    def __init__(self, model_path: str, intra_op_threads: int = 2):
        self.model_path = model_path
        self.session: Optional[ort.InferenceSession] = None
        self.request_count = 0
        self.error_count = 0
        self.total_latency_ms = 0.0
        self._intra_op_threads = intra_op_threads

    def startup(self) -> None:
        """Load model — called once at server start."""
        self.session = build_production_session(
            self.model_path, intra_op_threads=self._intra_op_threads
        )

    def health(self) -> Dict[str, Any]:
        return readiness_check(self.session, self.model_path)

    def metrics(self) -> Dict[str, Any]:
        avg_latency = (
            self.total_latency_ms / self.request_count
            if self.request_count > 0
            else 0.0
        )
        return {
            "total_requests": self.request_count,
            "total_errors": self.error_count,
            "avg_latency_ms": round(avg_latency, 3),
            "error_rate": round(
                self.error_count / max(self.request_count, 1), 4
            ),
        }

    def predict(self, request: Dict[str, Any]) -> Dict[str, Any]:
        """Full inference pipeline with metrics tracking."""
        self.request_count += 1
        start = time.perf_counter()

        try:
            response = handle_inference_request(self.session, request)
            latency = (time.perf_counter() - start) * 1000
            self.total_latency_ms += latency
            response["latency_ms"] = round(latency, 3)
            return response
        except InferenceError as e:
            self.error_count += 1
            return {"error": e.detail, "status_code": e.status_code}


# Test the complete server
server = ONNXInferenceServer(MODEL_PATH)
server.startup()

# Health check
health = server.health()
assert health["status"] == "ready"

# Successful predictions
for i in range(10):
    req = {"data": np.random.randn(4).tolist(), "shape": [1, 4]}
    resp = server.predict(req)
    assert "outputs" in resp

# Intentional error
bad_resp = server.predict({"data": [1], "shape": [99, 99]})
assert "error" in bad_resp

# Metrics
metrics = server.metrics()
print("Server metrics:", json.dumps(metrics, indent=2))
assert metrics["total_requests"] == 11
assert metrics["total_errors"] == 1
assert metrics["avg_latency_ms"] > 0
print("\nInference API server skeleton validated successfully!")

<a id='summary'></a>
## Summary

In this notebook we built all the components needed for cloud deployment:

| Component | Key Takeaway |
|-----------|-------------|
| **Production Session** | Thread tuning + provider selection for latency control |
| **Metadata Introspection** | Drives API schema and input validation |
| **Request Handler** | Validates, reshapes, infers, serializes |
| **Pre/Post Processing** | Numerically stable softmax, top-k extraction |
| **K8s Manifest** | Programmatic infra-as-code for deployment |
| **Health Probes** | Liveness vs readiness semantics |
| **Load Testing** | P50/P95/P99 latency → capacity planning |

**Next step:** wrap these handlers in a real FastAPI app and deploy behind an ingress controller.